#### Student Name:
#### Student ID:

# Diffusion with Stable Audio Open

Large generative models are not practical to train from scratch inside this course homework, so this assignment focuses on **sampling from a pretrained diffusion model** and on inference-time controls that make generation more creative and steerable.

The case-study model is **Stable Audio Open (SAO)**. Each question builds on the previous one, so complete the notebook in order:

1. implement a simple Euler sampler for reverse diffusion;
2. build an inpainting mask in latent time;
3. use a reference latent and a mask for audio inpainting;
4. control which diffusion steps apply the inpainting constraint;
5. implement style transfer by starting from a partially noised reference latent.

All required correctness checks happen in the **latent space of the autoencoder**. You may decode latents into audio, but audio playback cells are optional and are mainly for GPU users.

## Before you run

Run this notebook from the assignment folder. For the slim student handout, that folder only needs:

```text
homework4_diffusion_assignment.ipynb
requirements.txt
x_T.pt
testing_files/
```

Install the Python dependencies before the model-loading cell:

```bash
pip install -r requirements.txt
```

The notebook imports `stable_audio_tools` from the Python environment, so the handout does **not** need to carry a local `stable_audio_tools/` source tree or `setup.py` if `requirements.txt` installs `stable-audio-tools==0.0.19`.

This assignment uses the gated Hugging Face model:

```text
stabilityai/stable-audio-open-1.0
```

Before running the model-loading cell:

1. create or use a Hugging Face account;
2. open the model page and accept/request access;
3. create a read token;
4. set the token in your terminal, DataHub session, or Colab secret manager:

```bash
export HF_TOKEN='paste-your-token-here'
```

Do **not** paste your token into a notebook cell, and do not submit a token.

A GPU is helpful for iteration and audio playback. CPU is acceptable for the latent tests but can be slow. The Stable Audio model cache is several GB, so make sure your environment has enough available storage.


## How to use this notebook

Each numbered section has the same pattern:

1. read the short conceptual description,
2. fill in only the `# TODO` code for that question,
3. run the provided latent-space test, and
4. check that the printed latent MSE is low.

Optional audio cells are commented out. They are meant for exploration only and should stay commented when exporting code for submission unless your instructor says otherwise.


# Setup

The next cell imports the libraries, loads Stable Audio Open, and initializes the global constants used by the rest of the notebook:

- `SAMPLE_RATE`: model audio sample rate;
- `SAMPLE_SIZE`: latent-generation length used by the assignment tests;
- `SEED`: fixed random seed for repeatable latent checks;
- `device`: CUDA if available, otherwise CPU.

Keep `USE_HALF = False` while working on the required latent MSE checks. Half precision is only for optional exploratory audio generation after the tests pass.

In [ ]:
import os
import gc
import torch
import torchaudio
from einops import rearrange
from stable_audio_tools import get_pretrained_model
import IPython.display as ipd
from tqdm.auto import trange, tqdm
from stable_audio_tools.inference.generation import generate_diffusion_cond_and_sampler_setup, generate_diffusion_cond_decode

# Keep full precision for the reference latent tests. This is slower than half precision
# but makes CPU/GPU behavior easier to compare. Use half precision only for optional
# exploratory audio generation after the tests pass.
USE_HALF = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={device}")
print(f"HF token visible: {bool(os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_HUB_TOKEN'))}")

model, model_config = get_pretrained_model("stabilityai/stable-audio-open-1.0")
SAMPLE_RATE = model_config["sample_rate"]
SAMPLE_SIZE = model_config["sample_size"] // 8
SEED = 456

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

if USE_HALF:
    model = model.half()
model = model.to(device)

print(f"sample_rate={SAMPLE_RATE}")
print(f"sample_size={model_config['sample_size']}")
print(f"use_half={USE_HALF}")

### Colab / DataHub note

The next code cell is only a commented setup template. If your environment is already prepared, leave it commented. If you are running in Colab, uncomment the lines you need and make sure the working directory is the slim assignment folder above.


In [ ]:
# Colab-only setup (uncomment and edit the path if you are running from Google Drive).
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/path/to/diffusion_student_ver
# %pip install -r requirements.txt
# If Colab dependency resolution complains, the following pins have helped in some runtimes:
# %pip install numpy==1.26.4 protobuf==3.20.1


# Q1 - Simple diffusion sampler

Diffusion models are unusual among generative models because training and inference are algorithmically different. Even with a pretrained diffusion model, we still need to design a **sampler**: an iterative procedure that moves a noisy sample toward the data distribution.

Q1 has two pieces.

## Q1-1: `to_d()`

SAO uses **x-prediction**, meaning the model output is an estimate of the final clean sample. Sampling, however, needs a direction/derivative for the current noisy state. Your job is to convert the model's denoised prediction into that direction.

**Inputs**

- `x`: current noisy intermediate tensor, also the input to SAO at this step.
- `sigma`: current noise level.
- `denoised`: model prediction of the clean sample, with the same shape as `x`.

**Output**

Return a tensor with the same shape as `x` and `denoised`.

**Implementation guidance**

This should be short. Think about the denoising/score-matching connection from lecture: the denoised prediction and the current noisy point determine the direction used by the sampler.

## Q1-2: `simple_sample()`

Now use `to_d()` inside a simple Euler sampler.

At each diffusion step:

1. call the denoiser as `model(x, sigma * s_in, **extra_args)`,
2. convert the denoised prediction to a direction with `to_d`,
3. compute the step size from the current noise level and the next noise level, and
4. update `x` by stepping along the direction.
5. repeat until the final sigma is reached.

**Inputs**

- `model`: the SAO denoiser returned by the generation setup helper.
- `x`: the initial full Gaussian-noise latent.
- `sigmas`: descending noise schedule; usually `sigmas[i + 1] < sigmas[i]`.
- `extra_args`: conditioning arguments such as text guidance.

The starter code already creates `s_in` so the scalar noise level can be passed to the model as a batch-shaped tensor. The body of the loop can be short; focus on the update logic.

In [ ]:

def to_d(x, sigma, denoised):
    # TODO
    pass

@torch.no_grad()
def simple_sample(model, x, sigmas, extra_args=None):
    extra_args = {} if extra_args is None else extra_args
    s_in = x.new_ones([x.shape[0]])
    for i in trange(len(sigmas) - 1):
        # TODO: add extra_args
    del extra_args
    torch.cuda.empty_cache()
    return x

### Q1 generation helper

`generate()` sets up text conditioning, gets the denoiser/noise schedule, calls your `simple_sample()`, and optionally decodes latents to audio.

You normally do not need to edit this helper. Focus on the TODOs in `to_d()` and `simple_sample()`.

In [ ]:
def generate(prompt="128 BPM electronic drum loop", steps=50, cfg_scale=7, return_latents=False, x_start=None):

    # Set up text and timing conditioning
    conditioning = [{
        "prompt": prompt,
        "seconds_start": 0, 
        "seconds_total": 5
    }]

    # Generate diffusion setup params
    denoiser, x_T, sigmas, extra_args = generate_diffusion_cond_and_sampler_setup(
        model,
        steps=steps, # number of steps, more = better quality
        cfg_scale=cfg_scale, # Classifier-Free Guidance Scale, higher = better text relevance / quality but less diversity
        conditioning=conditioning,
        sample_size=SAMPLE_SIZE, # number of audio samples to generate, DON'T CHANGE
        device=device, # cuda device
        seed=SEED # random seed, DON'T CHANGE
    )

    if x_start is not None:
        x_T = x_start

    # Sample
    samples = simple_sample(denoiser, x_T, sigmas, extra_args=extra_args)
    del x_T
    del sigmas
    del extra_args
    torch.cuda.empty_cache()
    gc.collect()

    if return_latents:
        return samples

    # Decode
    audio = generate_diffusion_cond_decode(
        model,
        samples
    ).cpu()
    return audio



### Latent MSE check

Run the check below after implementing the current question. The reference tensors in `testing_files/` were generated by the expected latent behavior. Low MSE means your implementation is matching the reference.

In [ ]:
# to test your function, we provide some example latents to compare against
# the MSE between your latents and the reference latents should be low
for ix, prompt in enumerate(["lo-fi jazz piano in a rainy cafe", "deep ambient wash with ocean sounds"]):
    # load reference from testing_files
    ref = torch.load(f"testing_files/q1_{ix}.pt").to(device)

    # load x_T
    x_T = torch.load(f"x_T.pt").to(device)

    latents = generate(prompt=prompt, steps=50, cfg_scale=7, return_latents=True, x_start=x_T)
    # compare latents
    print(f"Latent MSE: {torch.nn.functional.mse_loss(ref, latents)}")


### Optional audio playback

The following cell is commented out because decoding and playback can be slow on CPU. Use it only after the latent checks pass.

In [ ]:
# for those running with a GPU, you can also generate audio samples
# this is impractical for CPU, but you can try it if you want (we DO NOT recommend it)
# for ix, prompt in enumerate(["lo-fi jazz piano in a rainy cafe", "deep ambient wash with ocean sounds"]):
#     # load reference from testing_files
#     audio = generate(prompt="128 BPM electronic drum loop", steps=50, cfg_scale=7, return_latents=False)
#     # play audio
#     ipd.display(ipd.Audio(audio.cpu().numpy()[0], rate=SAMPLE_RATE))

# Q2 - Inpainting mask

Now that you have a sampler, the next step is **inpainting**: keep part of a reference audio fixed and ask the model to fill in a masked region.

Before sampling with a mask, we need a helper that builds the mask in latent time.

## Implement `generate_inpainting_mask()`

**Goal**

Create a mask with the same shape as the reference latent. The mask should be:

- `1` where the model is allowed to inpaint;
- `0` where the reference should be preserved.

**Inputs**

- `reference`: encoded reference latent audio.
- `mask_start_s`: mask start time in seconds.
- `mask_end_s`: mask end time in seconds.

**Useful constants**

- `SAMPLE_RATE`: audio sample rate.
- `model.pretransform.downsampling_ratio`: how much SAO's encoder downsamples audio into latent frames.

**Implementation guidance**

The mask boundaries are specified in seconds, but the mask is applied in latent space. Convert seconds to audio samples, then account for the pretransform downsampling ratio to locate the corresponding latent-frame indices.

### Reference audio helpers

These helpers either load and encode a waveform or load an already encoded latent. The required tests use the provided latent tensors so students do not need to decode or submit audio.

In [ ]:
# LOAD AND ENCODE REFERENCE AUDIO
def load_and_encode_audio(path, model):
    audio, sr = torchaudio.load(path)
    resampler = torchaudio.transforms.Resample(sr, SAMPLE_RATE)
    audio = resampler(audio)
    audio = audio / audio.abs().max()

    if audio.shape[1] < SAMPLE_SIZE:
        while audio.shape[1] < SAMPLE_SIZE:
            audio = torch.cat((audio, audio), dim=1)

    audio = audio[:, :SAMPLE_SIZE][None].to(device)
    if USE_HALF:
        audio = audio.half()
    reference = model.pretransform.encode(audio)
    return reference


def load_encoded_audio(path):
    encoded_latent = torch.load(path)
    encoded_latent = encoded_latent.to(device)
    if USE_HALF:
        encoded_latent = encoded_latent.half()
    return encoded_latent


In [ ]:
def generate_inpainting_mask(reference, mask_start_s, mask_end_s):
    # TODO
    return mask

### Latent MSE check

Run the check below after implementing the current question. The reference tensors in `testing_files/` were generated by the expected latent behavior. Low MSE means your implementation is matching the reference.

In [ ]:
# to test your function, we provide some example latents to compare against
# the MSE between your latents and the reference latents should be low
for ix in range(2):
    for midx, mask_range in enumerate([(0,4), (1,2), (1.5,3), (2,4), (1,4), (0,5)]):
        # load reference from testing_files
        ref = torch.load(f"testing_files/q2_{ix}.pt")[midx].to(device)
        # load reference audio
        reference = load_encoded_audio(f"testing_files/q1_{ix}.pt")
        # generate mask
        mask = generate_inpainting_mask(reference, *mask_range)
        # compare latents
        print(f"Latent MSE: {torch.nn.functional.mse_loss(ref, mask)}")



# Q3 - Inpainting sampler

Inpainting reuses the Q1 sampler with one extra constraint. After each sampling update, the current latent should be combined with a noisy version of the reference latent:

1. add the right amount of Gaussian noise to the reference so it matches the current diffusion noise level;
2. use the mask so the inpainting region remains generated by the sampler while the unmasked region follows the noisy reference.

## Implement `simple_sample_inpaint()`

**Inputs**

- `model`: SAO denoiser.
- `x`: initial noisy latent.
- `sigmas`: noise schedule.
- `reference`: encoded reference audio.
- `mask`: mask from `generate_inpainting_mask()`.
- `extra_args`: conditioning arguments.

This is a small modification of your Q1 sampler. Keep the sampling logic the same and add the inpainting constraint inside the loop.

In [ ]:
@torch.no_grad()
def simple_sample_inpaint(model, x, sigmas, reference, mask, extra_args=None):
    extra_args = {} if extra_args is None else extra_args
    s_in = x.new_ones([x.shape[0]])
    for i in trange(len(sigmas) - 1):
        # TODO
    del extra_args
    torch.cuda.empty_cache()
    return x


### Q3 inpainting helper

`inpaint()` creates the text conditioning, builds the mask, prepares the diffusion schedule, calls your inpainting sampler, and optionally decodes the result.

In [ ]:
def inpaint(prompt="128 BPM house drum loop", steps=50, cfg_scale=7, reference=None, mask_start_s=20, mask_end_s=30, return_latents=False, x_start=None):
    # Set up text and timing conditioning
    conditioning = [{
        "prompt": prompt,
        "seconds_start": 0, 
        "seconds_total": 5
    }]
    # Set up inpainting mask
    mask = generate_inpainting_mask(reference, mask_start_s, mask_end_s)

    # Generate diffusion setup params
    denoiser, x_T, sigmas, extra_args = generate_diffusion_cond_and_sampler_setup(
        model,
        steps=steps,
        cfg_scale=cfg_scale,
        conditioning=conditioning,
        sample_size=SAMPLE_SIZE,
        device=device,
        seed=SEED
    )

    if x_start is not None:
        x_T = x_start

    # Sample
    inp_samples = simple_sample_inpaint(denoiser, x_T, sigmas, reference, mask, extra_args=extra_args)
    del x_T
    del sigmas
    del extra_args
    torch.cuda.empty_cache()
    gc.collect()

    if return_latents:
        return inp_samples

    # decode and play
    inpainted_audio = generate_diffusion_cond_decode(
        model,
        inp_samples
    ).cpu()
    return inpainted_audio



### Latent MSE check

Run the check below after implementing the current question. The reference tensors in `testing_files/` were generated by the expected latent behavior. Low MSE means your implementation is matching the reference.

In [ ]:
# to test your function, we provide some example latents to compare against
# the MSE between your latents and the reference latents should be low

for ix, prompt in enumerate(["lo-fi jazz piano in a rainy cafe", "deep ambient wash with ocean sounds"]):
    # load reference from testing_files
    ref = torch.load(f"testing_files/q3_{ix}.pt").to(device)
    # load reference audio
    reference = load_encoded_audio(f"testing_files/q1_{ix}.pt")
    # generate mask
    mask = generate_inpainting_mask(reference, 0, 3)

    # load x_T
    x_T = torch.load(f"x_T.pt").to(device)

    # generate inpainting
    latents = inpaint(prompt=prompt, steps=50, cfg_scale=7, reference=reference, mask_start_s=0, mask_end_s=3, return_latents=True, x_start=x_T)
    # compare latents
    print(f"Latent MSE: {torch.nn.functional.mse_loss(ref, latents)}")


### Optional audio playback

The following cell is commented out because decoding and playback can be slow on CPU. Use it only after the latent checks pass.

In [ ]:
# for those running with a GPU, you can also generate audio samples
# this is impractical for CPU, but you can try it if you want (we DO NOT recommend it)
# for ix, prompt in enumerate(["lo-fi jazz piano in a rainy cafe", "deep ambient wash with ocean sounds"]):
#     # load reference from testing_files
#     reference = load_encoded_audio(f"testing_files/q1_{ix}.pt")
#     # generate mask
#     mask = generate_inpainting_mask(reference, 0, 3)
#     # generate inpainting
#     audio = inpaint(prompt=prompt, steps=50, cfg_scale=7, reference=reference, mask_start_s=0, mask_end_s=3, return_latents=False)
#     # play audio
#     ipd.display(ipd.Audio(audio.cpu().numpy()[0], rate=SAMPLE_RATE))


# Q4 - Variable-strength inpainting

Full inpainting at every step can sometimes make the transition at the mask boundary feel abrupt. In Q4, you will apply the inpainting constraint only for a selected range of diffusion steps.

## Implement `simple_sample_variable_inpaint()`

This should behave like `simple_sample_inpaint()` except that the inpainting operation is only applied when the current step index lies in the requested range.

**Inputs**

- `model`: SAO denoiser.
- `x`: initial noisy latent.
- `sigmas`: noise schedule.
- `reference`: encoded reference audio.
- `mask`: mask from `generate_inpainting_mask()`.
- `paint_start`: first step index where inpainting is applied.
- `paint_end`: last step index where inpainting is applied.
- `extra_args`: conditioning arguments.

When `paint_start = 0` and `paint_end = len(sigmas) - 1`, this function should match your Q3 inpainting sampler.

In [ ]:
@torch.no_grad()
def simple_sample_variable_inpaint(model, x, sigmas, reference, mask, extra_args=None, paint_start=None, paint_end=None):
    if paint_start is None:
        paint_start = 0
    if paint_end is None:
        paint_end = len(sigmas) - 1
    extra_args = {} if extra_args is None else extra_args
    s_in = x.new_ones([x.shape[0]])
    for i in trange(len(sigmas) - 1):
        # TODO
    del extra_args
    torch.cuda.empty_cache()
    return x


### Q4 helper

`variable_inpaint()` is a wrapper around your variable inpainting sampler. The tests below use the same mask range and vary only the step range where the inpainting constraint is active.

In [ ]:
def variable_inpaint(prompt="128 BPM house drum loop", steps=50, cfg_scale=7, reference=None, mask_start_s=20, mask_end_s=30, paint_start=None, paint_end=None, return_latents=False, x_start=None):
    # Set up text and timing conditioning
    conditioning = [{
        "prompt": prompt,
        "seconds_start": 0, 
        "seconds_total": 5
    }]
    # Set up inpainting mask
    mask = generate_inpainting_mask(reference, mask_start_s, mask_end_s)

    # Generate diffusion setup params
    denoiser, x_T, sigmas, extra_args = generate_diffusion_cond_and_sampler_setup(
        model,
        steps=steps,
        cfg_scale=cfg_scale,
        conditioning=conditioning,
        sample_size=SAMPLE_SIZE,
        device=device,
        seed=SEED
    )

    if x_start is not None:
        x_T = x_start

    # Sample
    inp_samples = simple_sample_variable_inpaint(denoiser, x_T, sigmas, reference, mask, extra_args=extra_args, paint_start=paint_start, paint_end=paint_end)
    del x_T
    del sigmas
    del extra_args
    torch.cuda.empty_cache()
    gc.collect()

    if return_latents:
        return inp_samples

    # decode and play
    inpainted_audio = generate_diffusion_cond_decode(
        model,
        inp_samples
    ).cpu()
    return inpainted_audio



### Latent MSE check

Run the check below after implementing the current question. The reference tensors in `testing_files/` were generated by the expected latent behavior. Low MSE means your implementation is matching the reference.

In [ ]:
# to test your function, we provide some example latents to compare against
# the MSE between your latents and the reference latents should be low

for ix, prompt in enumerate(["lo-fi jazz piano in a rainy cafe", "deep ambient wash with ocean sounds"]):
    ref = torch.load(f"testing_files/q4_{ix}.pt").to(device)
    if USE_HALF:
        ref = ref.half()

    reference = load_encoded_audio(f"testing_files/q1_{ix}.pt")

    # The reference files were generated with the same mask range (0, 3) for both prompts.
    # Q4 varies when inpainting is applied, not the mask range itself.
    mask_start_s = 0
    mask_end_s = 3
    if ix == 0:
        paint_start = 0
        paint_end = 20
    else:
        paint_start = 15
        paint_end = 45

    x_T = torch.load(f"x_T.pt").to(device)
    if USE_HALF:
        x_T = x_T.half()

    latents = variable_inpaint(
        prompt=prompt,
        steps=50,
        cfg_scale=7,
        reference=reference,
        mask_start_s=mask_start_s,
        mask_end_s=mask_end_s,
        paint_start=paint_start,
        paint_end=paint_end,
        return_latents=True,
        x_start=x_T
    )
    print(f"Latent MSE: {torch.nn.functional.mse_loss(ref.float(), latents.float())}")


### Optional audio playback

The following cell is commented out because decoding and playback can be slow on CPU. Use it only after the latent checks pass.

In [ ]:
# for those running with a GPU, you can also generate audio samples
# this is impractical for CPU, but you can try it if you want (we DO NOT recommend it)
# for ix, prompt in enumerate(["lo-fi jazz piano in a rainy cafe", "deep ambient wash with ocean sounds"]):
#     # load reference from testing_files
#     reference = load_encoded_audio(f"testing_files/q1_{ix}.pt")
#     # generate mask
#     mask = generate_inpainting_mask(reference, 0, 3)
#     # generate inpainting
#     if ix == 0:
#         paint_start = 0
#         paint_end = 20
#     else: 
#         paint_start = 15
#         paint_end = 45    
#     audio = variable_inpaint(prompt=prompt, steps=50, cfg_scale=7, reference=reference, mask_start_s=0, mask_end_s=3, paint_start=paint_start, paint_end=paint_end, return_latents=False)
#     # play audio
#     ipd.display(ipd.Audio(audio.cpu().numpy()[0], rate=SAMPLE_RATE))

# Q5 - Style transfer

Finally, you will implement a simple diffusion-based style transfer. Instead of masking a local region, style transfer is a global operation:

1. start from an encoded reference latent;
2. add noise up to a strength-controlled level;
3. run the normal sampler from that partially noised reference.

A small `transfer_strength` should mostly ignore the reference. A large `transfer_strength` should make the output stay closer to the reference.

## Implement `simple_sample_style_transfer()`

**Inputs**

- `model`: SAO denoiser.
- `sigmas`: noise schedule.
- `reference`: encoded reference audio.
- `extra_args`: conditioning arguments.
- `transfer_strength`: float from `0` to `1`.
- `x_start`: optional deterministic noise tensor supplied by the wrapper/tests.

**Implementation guidance**

Map `transfer_strength` to an index or noise level in `sigmas`, construct the partially noised reference, and then sample from that point. You may reuse your Q1 sampler directly after choosing the correct starting state and suffix of the noise schedule.

In [ ]:
def simple_sample_style_transfer(model, sigmas, reference, extra_args=None, transfer_strength=0, x_start=None):
    # TODO
    pass


### Q5 helper

`style_transfer()` sets up conditioning and passes the deterministic `x_T` noise tensor into your style-transfer implementation for reproducible latent checks.

In [ ]:
def style_transfer(prompt="128 BPM house drum loop", steps=50, cfg_scale=7, reference=None, transfer_strength=0, return_latents=False, x_start=None):
    # Set up text and timing conditioning
    conditioning = [{
        "prompt": prompt,
        "seconds_start": 0, 
        "seconds_total": 5
    }]

    # Generate diffusion setup params
    denoiser, x_T, sigmas, extra_args = generate_diffusion_cond_and_sampler_setup(
        model,
        steps=steps,
        cfg_scale=cfg_scale,
        conditioning=conditioning,
        sample_size=SAMPLE_SIZE,
        device=device,
        seed=SEED
    )
    if x_start is not None:
        x_T = x_start

    # Sample
    inp_samples = simple_sample_style_transfer(denoiser, sigmas, reference, extra_args=extra_args, transfer_strength=transfer_strength, x_start=x_T)
    del x_T
    del sigmas
    del extra_args
    torch.cuda.empty_cache()
    gc.collect()

    if return_latents:
        return inp_samples
    
    # decode and play
    inpainted_audio = generate_diffusion_cond_decode(
        model,
        inp_samples
    ).cpu()
    return inpainted_audio
    

### Latent MSE check

Run the check below after implementing the current question. The reference tensors in `testing_files/` were generated by the expected latent behavior. Low MSE means your implementation is matching the reference.

In [ ]:
# to test your function, we provide some example latents to compare against
# the MSE between your latents and the reference latents should be low
for ix, prompt in enumerate(["deep ambient wash with ocean sounds", "lo-fi jazz piano in a rainy cafe"]):
    # load reference from testing_files
    ref = torch.load(f"testing_files/q5_{ix}.pt").to(device)
    # load reference audio
    reference = load_encoded_audio(f"testing_files/q1_{ix}.pt")
    if ix == 0:
        transfer_strength = 0.2
    else:
        transfer_strength = 0.5

    # load x_T
    x_T = torch.load(f"x_T.pt").to(device)

    # generate style transfer
    latents = style_transfer(prompt=prompt, steps=50, cfg_scale=7, reference=reference, transfer_strength=transfer_strength, return_latents=True, x_start=x_T)
    # compare latents
    print(f"Latent MSE: {torch.nn.functional.mse_loss(ref, latents)}")

### Optional audio playback

The following cell is commented out because decoding and playback can be slow on CPU. Use it only after the latent checks pass.

In [ ]:
# for those running with a GPU, you can also generate audio samples
# this is impractical for CPU, but you can try it if you want (we DO NOT recommend it)
# for ix, prompt in enumerate(["lo-fi jazz piano in a rainy cafe", "deep ambient wash with ocean sounds"]):
#     # load reference from testing_files
#     reference = load_encoded_audio(f"testing_files/q1_{ix}.pt")
#     if ix == 0:
#         transfer_strength = 0.2
#     else:
#         transfer_strength = 0.5
#     # generate style transfer
#     audio = style_transfer(prompt=prompt, steps=50, cfg_scale=7, reference=reference, transfer_strength=transfer_strength, return_latents=False)
#     # play audio
#     ipd.display(ipd.Audio(audio.cpu().numpy()[0], rate=SAMPLE_RATE))

# Finishing checklist

Before submitting, make sure:

- all TODOs in Q1-Q5 are implemented;
- the latent MSE checks for each question are low;
- audio playback cells are still commented out or skipped if your environment is CPU-only;
- no Hugging Face token appears anywhere in the notebook.

Submit according to the course/Gradescope instructions for this assignment.